# SML-MFD | 60-Month Forecast Accuracy Analysis

**Research project:** Machine Forecast Disagreement (SML)  
**Author:** Daehyun Kim | **Analysis version:** November 2026  
**Data:** Team-produced `accuracy_overall_60.csv` and `accuracy_monthly_60.csv`  
**Sample:** Forecast signal months January 2005–November 2015 (131 observed months)

This notebook reproduces the out-of-sample accuracy analysis delivered to the SML team. It compares **accounting** and **market** forecasts on squared-error accuracy and monthly cross-sectional stock-return ranking accuracy.

**Reproducibility note:** The CSV files already contain results from the team's full model run. This notebook *analyzes those results*; it does **not** retrain Random Forests, run 1,100 SLURM jobs, or backtest a trading strategy.

## 0. Analysis roadmap

1. Import two team CSVs and validate their contents.  
2. Compare overall RMSE, zero benchmark, out-of-sample R-squared, and monthly mean Spearman IC.  
3. Plot monthly IC, three-month moving-average IC, and annual mean IC.  
4. Identify runs of negative IC and the five strongest/weakest months.  
5. Check sensitivity to extreme months.  
6. Conduct a paired monthly IC t-test, plus an **optional** autocorrelation-robust sensitivity check.  
7. Save publication-ready PNG figures and CSV tables to `outputs/`.

**Key terms:** `eom` is the signal month; `spearman_ic` is the cross-sectional correlation between predicted and realized next-month returns. Values above 0 signal better-than-random *ranking on average*, not necessarily better squared-error forecasts or profitable portfolios.

In [ ]:
# Cell 1 — Imports and output folder
from pathlib import Path
from io import BytesIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:.6f}")
plt.rcParams.update({
    "figure.dpi": 115, "savefig.dpi": 200,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 13,
})
print("Output directory:", OUTPUT_DIR.resolve())

## 1. Load the official team CSV files

**In Google Colab:** Run the next cell and upload *both* `accuracy_overall_60.csv` and `accuracy_monthly_60.csv` together. If Colab has saved older files as `... (1).csv`, this code still uses the **newly uploaded bytes**, avoiding accidental use of stale copies.

**On a local computer:** Place the two CSV files in the current working directory and run the cell. Download them from the team Google Drive folder when necessary: [mfd_60_month_results_20261030](https://drive.google.com/drive/folders/1atJkvcJmcijm4vngJSEQF_D8HFRIMv_9).

In [ ]:
# Cell 2 — Colab upload (with a local-Python fallback)
try:
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    def uploaded_csv(stem):
        matching = [name for name in uploaded
                    if name.startswith(stem) and name.lower().endswith(".csv")]
        if len(matching) != 1:
            raise ValueError(f"Please upload exactly one CSV matching {stem!r}; found {matching}")
        return pd.read_csv(BytesIO(uploaded[matching[0]]))
    overall = uploaded_csv("accuracy_overall_60")
    monthly = uploaded_csv("accuracy_monthly_60")
except ImportError:
    overall = pd.read_csv("accuracy_overall_60.csv")
    monthly = pd.read_csv("accuracy_monthly_60.csv")

monthly["eom"] = pd.to_datetime(monthly["eom"], errors="raise")
monthly = monthly.sort_values(["track", "eom"]).reset_index(drop=True)
overall = overall.sort_values("track").reset_index(drop=True)
print("Overall rows:", len(overall), "| Monthly rows:", len(monthly))
print("Forecast dates:", monthly.eom.min().date(), "through", monthly.eom.max().date())
display(overall)

### Data quality checks

Expected rows: 2 overall rows (one per track) and 262 monthly rows (131 × 2). Both tracks should have the same 131 dates and 308,348 forecast evaluations. The final month is November 2015 because the next-month outcome for December 2015 is not available in this dataset.

In [ ]:
# Cell 3 — Structural and numerical validation
tracks = {"accounting", "market"}
assert set(overall.track) == tracks and set(monthly.track) == tracks
assert len(overall) == 2
assert monthly.groupby("track").size().to_dict() == {"accounting": 131, "market": 131}
assert monthly.groupby("track").eom.nunique().eq(131).all()
assert monthly.eom.max() == pd.Timestamp("2015-11-30")
assert monthly.groupby("track").n_predictions.sum().to_dict() == {
    "accounting": 308348, "market": 308348
}
assert monthly.pivot(index="eom", columns="track", values="spearman_ic").dropna().shape == (131, 2)
assert np.isfinite(monthly["spearman_ic"]).all()

for _, row in overall.iterrows():
    subset = monthly.loc[monthly.track == row["track"], "spearman_ic"]
    assert np.isclose(subset.mean(), row["mean_monthly_spearman_ic"], atol=1e-12)
    assert np.isclose((subset > 0).mean(), row["positive_ic_month_fraction"], atol=1e-12)
print("PASS: file structure, paired dates, prediction counts, and overall IC summaries match.")

## 2. Overall forecast accuracy

**Root mean squared error (RMSE):** average prediction error, giving extra weight to large misses; **lower** is better.  
**Zero benchmark:** predicts a return of zero for every stock.  
**Out-of-sample R-squared:** \(R^2_{OOS}=1-\frac{\sum (r-\hat r)^2}{\sum r^2}\). Negative values mean the model has a larger sum of squared errors than the zero benchmark.  
**Mean monthly Spearman IC:** the average, over months, of the cross-sectional Spearman rank correlation; **higher** is better for ordering stocks.

Use the official `accuracy_overall_60.csv` for reported overall RMSE and R-squared. *Do not average monthly RMSEs to recreate pooled RMSE*: the pooled statistic requires prediction-count weighting of squared errors.

In [ ]:
# Cell 4 — Overall comparison table (official report values)
metrics = ["n_predictions", "rmse", "zero_benchmark_rmse", "oos_r_squared",
           "mean_monthly_spearman_ic", "positive_ic_month_fraction", "simple_ic_t_stat"]
comparison = overall.set_index("track")[metrics].T
comparison.index.name = "metric"
print(comparison.round(6).to_string())
comparison.to_csv(OUTPUT_DIR / "overall_comparison.csv")
print("\nInterpretation:")
for _, row in overall.iterrows():
    better = row.rmse < row.zero_benchmark_rmse
    print(f"- {row.track.capitalize()}: RMSE {row.rmse:.6f} vs zero benchmark "
          f"{row.zero_benchmark_rmse:.6f}; OOS R² = {row.oos_r_squared:.6f}; "
          f"mean monthly IC = {row.mean_monthly_spearman_ic:.6f}. "
          f"RMSE benchmark {'beaten' if better else 'NOT beaten'}.")

## 3. Monthly Spearman IC over time

A single average can hide periods when the model's stock ranking does poorly. Plot the **unmodified monthly observations**, plus a horizontal zero line. Each monthly IC is computed across stocks, not across time.

In [ ]:
# Cell 5 — Monthly IC, both tracks
fig, ax = plt.subplots(figsize=(12.8, 4.5))
for track in ("accounting", "market"):
    s = monthly.loc[monthly.track == track]
    ax.plot(s.eom, s.spearman_ic, linewidth=1.25, alpha=0.85, label=track.capitalize())
ax.axhline(0, color="gray", linewidth=1, linestyle="--")
ax.set(title="Monthly Spearman IC | Accounting vs Market (2005–2015)",
       xlabel="Forecast signal month", ylabel="Spearman IC")
ax.legend(loc="upper left", frameon=False)
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_monthly_spearman_ic.png", bbox_inches="tight")
plt.show()

## 4. Three-month rolling IC and annual means

A rolling mean reduces month-to-month noise and highlights stretches of sustained weak performance. We use **three complete months** (`min_periods=3`), leaving the first two months unplotted. These are descriptive summaries—not a model or trading signal.

In [ ]:
# Cell 6 — Three-month moving-average IC
monthly["rolling_3m_ic"] = (
    monthly.groupby("track", sort=False)["spearman_ic"]
    .transform(lambda s: s.rolling(window=3, min_periods=3).mean())
)
fig, ax = plt.subplots(figsize=(12.8, 4.5))
for track in ("accounting", "market"):
    s = monthly.loc[monthly.track == track]
    ax.plot(s.eom, s.rolling_3m_ic, linewidth=1.65, label=track.capitalize())
ax.axhline(0, color="gray", linewidth=1, linestyle="--")
ax.set(title="Three-Month Rolling Average of Spearman IC",
       xlabel="Forecast signal month", ylabel="3-month average IC")
ax.legend(loc="upper left", frameon=False)
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_rolling_3m_ic.png", bbox_inches="tight")
plt.show()

In [ ]:
# Cell 7 — Year-by-year means
monthly["year"] = monthly.eom.dt.year
yearly = (monthly.pivot_table(index="year", columns="track", values="spearman_ic", aggfunc="mean")
          .sort_index())
print("Yearly mean Spearman IC:")
print(yearly.round(4).to_string())
yearly.to_csv(OUTPUT_DIR / "annual_mean_ic.csv")

fig, ax = plt.subplots(figsize=(10.5, 4.3))
for track in ("accounting", "market"):
    ax.plot(yearly.index, yearly[track], marker="o", linewidth=1.75, label=track.capitalize())
ax.axhline(0, color="gray", linewidth=1, linestyle="--")
ax.set(title="Annual Mean of Monthly Spearman IC", xlabel="Forecast year",
       ylabel="Mean monthly IC", xticks=yearly.index)
ax.legend(frameon=False)
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_annual_mean_ic.png", bbox_inches="tight")
plt.show()
print("Note: 2015 includes 11 months; December 2015 has no observed next-month label.")

## 5. Sustained weak periods and monthly extremes

Definition for this descriptive check: a **negative-IC streak** is a sequence of consecutive observed signal months with `spearman_ic < 0`. Zero and positive values break a streak. We compute streaks within each track in chronological order. This is stricter than describing a weak *annual average*.

In [ ]:
# Cell 8 — Detect every negative monthly-IC streak

def negative_ic_streaks(track_data):
    data = track_data.sort_values("eom").reset_index(drop=True)
    is_negative = data["spearman_ic"].lt(0).to_numpy()
    spans = []
    start = None
    for i, flag in enumerate(is_negative):
        if flag and start is None:
            start = i
        if start is not None and (not flag or i == len(data) - 1):
            end = i - 1 if not flag else i
            spans.append({
                "track": str(data.loc[0, "track"]),
                "start": data.loc[start, "eom"].strftime("%Y-%m"),
                "end": data.loc[end, "eom"].strftime("%Y-%m"),
                "months": end - start + 1,
                "mean_ic_during_streak": data.loc[start:end, "spearman_ic"].mean(),
            })
            start = None
    return pd.DataFrame(spans)

streaks = pd.concat(
    [negative_ic_streaks(monthly[monthly.track == track])
     for track in ("accounting", "market")], ignore_index=True
)
streaks = streaks.sort_values(["track", "months", "start"],
                              ascending=[True, False, True]).reset_index(drop=True)
streaks.to_csv(OUTPUT_DIR / "negative_ic_streaks.csv", index=False)
for track in ("accounting", "market"):
    print(f"\n{track.upper()}: 5 longest negative-IC streaks")
    print(streaks.loc[streaks.track == track].head(5).round(4).to_string(index=False))

In [ ]:
# Cell 9 — Identify high and low monthly IC observations
extremes = []
for track in ("accounting", "market"):
    s = monthly.loc[monthly.track == track, ["track", "eom", "spearman_ic"]]
    extremes.extend([s.nsmallest(5, "spearman_ic").assign(category="five_lowest"),
                     s.nlargest(5, "spearman_ic").assign(category="five_highest")])
extremes = pd.concat(extremes, ignore_index=True)
extremes.to_csv(OUTPUT_DIR / "monthly_ic_extremes.csv", index=False)
for track in ("accounting", "market"):
    print(f"\n{track.upper()}: five lowest IC months")
    print(extremes.query("track == @track and category == 'five_lowest'")
          [["eom", "spearman_ic"]].round(4).to_string(index=False))
    print(f"{track.upper()}: five highest IC months")
    print(extremes.query("track == @track and category == 'five_highest'")
          [["eom", "spearman_ic"]].round(4).to_string(index=False))

## 6. Robustness to unusually strong/weak months

The team previously examined two simple descriptive sensitivity checks:

- **Symmetric trimming:** remove the highest 5 and lowest 5 monthly IC observations *within each track*.
- **Positive-tail removal:** remove only the highest 5 monthly IC observations *within each track*.

The two methods answer different questions: symmetric trimming tests sensitivity to both tails; removing just the best months tests whether a few exceptionally successful forecasts account for the positive mean. They do **not** establish portfolio profitability or replace formal time-series inference.

In [ ]:
# Cell 10 — Reproduce both outlier checks
sensitivity_rows = []
for track, group in monthly.groupby("track"):
    ordered = group.spearman_ic.dropna().sort_values().to_numpy()
    if len(ordered) <= 10:
        raise ValueError("Need more than 10 months to trim 5 at each end.")
    sensitivity_rows.append({
        "track": track, "n_months": len(ordered),
        "full_mean_ic": np.mean(ordered),
        "trimmed_5_each_tail": np.mean(ordered[5:-5]),
        "without_top_5": np.mean(ordered[:-5]),
    })
sensitivity = pd.DataFrame(sensitivity_rows).set_index("track")
print(sensitivity.round(6).to_string())
sensitivity.to_csv(OUTPUT_DIR / "outlier_sensitivity.csv")

fig, ax = plt.subplots(figsize=(9.4, 4.2))
sensitivity[["full_mean_ic", "without_top_5"]].plot(kind="bar", ax=ax, rot=0)
ax.set(title="Mean Monthly Spearman IC | Excluding Five Best Months",
       xlabel="Forecast track", ylabel="Mean monthly IC")
ax.legend(["All months", "Excluding top 5"], frameon=False)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_outlier_robustness.png", bbox_inches="tight")
plt.show()

## 7. Accounting vs Market: paired monthly IC test

Because the models predict stocks in the **same calendar months**, the natural unit of comparison is the *monthly difference*:

\[d_t=IC_{accounting,t}-IC_{market,t}.\]

The null hypothesis is \(E[d_t]=0\). The **paired t-test** below reproduces the exploratory result already reported to the team (p ≈ 0.9271). A high p-value means **no statistically significant difference detected**; it does *not* prove the two models equivalent. The ordinary paired t-test assumes the differences are approximately independent across months; a HAC check is provided afterwards.

In [ ]:
# Cell 11 — Paired monthly IC test
paired = monthly.pivot(index="eom", columns="track", values="spearman_ic").dropna()
paired = paired.sort_index()
delta = paired["accounting"] - paired["market"]
stat, p_value = ttest_rel(paired["accounting"], paired["market"])
print(f"Paired months: {len(paired)}")
print(f"Accounting mean IC: {paired.accounting.mean():.6f}")
print(f"Market mean IC: {paired.market.mean():.6f}")
print(f"Mean IC difference (accounting - market): {delta.mean():.6f}")
print(f"Paired t-statistic: {stat:.4f}")
print(f"Two-sided p-value: {p_value:.4f}")
assert np.isclose(delta.mean(), 0.00065963402400355, atol=1e-10)
assert np.isclose(p_value, 0.9271, atol=1e-4)

pd.DataFrame({"date": paired.index, "accounting_ic": paired.accounting.to_numpy(),
              "market_ic": paired.market.to_numpy(), "difference_ic": delta.to_numpy()}) \
    .to_csv(OUTPUT_DIR / "paired_monthly_ic.csv", index=False)

### Optional: HAC / Newey–West sensitivity check

Forecast quality may be autocorrelated across adjacent months. A regression of monthly differences on a constant estimates the average difference; using HAC standard errors with 3 lags gives a simple robustness comparison. This is **supplementary** and may differ from the uncorrected paired t-test. Lag choice is a sensitivity setting, not a research result in itself.

In [ ]:
# Cell 12 — Optional time-series-robust mean-difference inference
try:
    import statsmodels.api as sm
    hac = sm.OLS(delta.to_numpy(), np.ones((len(delta), 1))).fit(
        cov_type="HAC", cov_kwds={"maxlags": 3})
    print(f"HAC (Newey-West, 3 lags) mean difference: {hac.params[0]:.6f}")
    print(f"HAC t/z statistic: {hac.tvalues[0]:.4f}")
    print(f"HAC two-sided p-value: {hac.pvalues[0]:.4f}")
except ImportError:
    print("Optional statsmodels package unavailable. In Colab: !pip install statsmodels")

## 8. Final observations and limitations

**Observed findings from the official CSVs**

- Both tracks have **negative OOS R²** and RMSE slightly **above the zero-return benchmark**: the models did not outperform the zero forecast on squared-error accuracy.
- Mean monthly Spearman IC is **about +0.036** in both tracks: rank associations are positive on average, but modest.
- Market had a **negative mean annual IC in 2009**; Accounting was close to zero that year. Accounting's longest run of negative monthly IC was **5 months (April–August 2005)**; Market's was **3 months**.
- Excluding each track's five strongest months leaves a **positive mean IC**. Paired comparison gives **p ≈ 0.9271**, so there is no detected difference between mean IC values in this unadjusted test.

**Limitations**

1. These statistics measure forecast errors and ranking, **not investment returns**. Task 7 portfolio backtesting is separate.
2. Positive IC does not prove economically profitable, implementable strategies.
3. The t-test is not a test of equivalence; monthly dependence warrants robust standard errors.
4. Weak periods and extremal months were identified descriptively, not through a causal design.
5. Predictions end **November 2015**, not December, because the next-month target is missing at the sample boundary.

**Data provenance:** Team-generated 60-month run outputs, distributed through the shared SML Google Drive folder. **Primary report:** `docs/SML_Forecast_Accuracy_Research_Report_EN.pdf`.

In [ ]:
# Cell 13 — Review generated files and optionally download them from Colab
print("Generated reproducibility artifacts:")
for file_path in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {file_path.name:35s} {file_path.stat().st_size:>8,d} bytes")

# Uncomment only when you wish to download every output in a single ZIP from Colab:
# import shutil
# from google.colab import files
# archive_path = shutil.make_archive("SML_Forecast_Accuracy_Outputs", "zip", OUTPUT_DIR)
# files.download(archive_path)